# PyTorch回调函数详解

> 本教程是 [Keras回调函数详解](./使用回调函数.ipynb) 的 PyTorch 等价版本。

PyTorch 没有像 Keras 那样内置的回调系统，但我们可以自己实现一个轻量级的回调框架，
同时利用 PyTorch 内置的 `torch.optim.lr_scheduler` 来实现学习率调度。

## 学习目标

1. 理解回调函数的工作机制
2. 掌握 PyTorch 中实现 ModelCheckpoint、EarlyStopping 等回调的方法
3. 学会使用 PyTorch 内置的 lr_scheduler（ReduceLROnPlateau、LambdaLR、CosineAnnealingLR）
4. 学会实现自定义回调
5. 了解回调函数的最佳实践

## 1. 环境配置与数据准备

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"PyTorch版本 / PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备 / Device: {device}")

In [ ]:
# 准备数据 / Prepare data
housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# 转换为PyTorch张量 / Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_valid_t = torch.FloatTensor(X_valid)
y_valid_t = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# 创建DataLoader / Create DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(X_valid_t, y_valid_t)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"训练集 / Training set: {X_train.shape[0]} 样本 / samples")
print(f"验证集 / Validation set: {X_valid.shape[0]} 样本 / samples")
print(f"测试集 / Test set: {X_test.shape[0]} 样本 / samples")

In [ ]:
def create_model():
    """
    创建回归模型 / Create a regression model

    等价于Keras的Sequential模型：
    Dense(30, relu) -> Dense(30, relu) -> Dense(1)

    Equivalent to Keras Sequential:
    Dense(30, relu) -> Dense(30, relu) -> Dense(1)
    """
    model = nn.Sequential(
        nn.Linear(8, 30),
        nn.ReLU(),
        nn.Linear(30, 30),
        nn.ReLU(),
        nn.Linear(30, 1)
    )
    return model.to(device)

model = create_model()
print(model)

## 2. 回调系统基础框架

PyTorch 没有内置的回调系统，我们需要自己实现。下面定义一个轻量级的回调基类和训练循环。

核心思路：
- 定义 `Callback` 基类，提供 `on_train_begin`、`on_epoch_end`、`on_train_end` 等钩子
- 在训练循环的适当位置调用这些钩子
- 各个具体回调（EarlyStopping、ModelCheckpoint等）继承基类并实现相应方法

In [ ]:
class Callback:
    """
    回调基类 / Base callback class

    提供训练过程中各阶段的钩子方法。
    子类可以重写需要的方法来实现自定义行为。

    Provides hook methods at various stages of training.
    Subclasses can override methods to implement custom behavior.
    """

    def on_train_begin(self, logs=None):
        """训练开始时调用 / Called at the beginning of training"""
        pass

    def on_epoch_begin(self, epoch, logs=None):
        """每个epoch开始时调用 / Called at the beginning of each epoch"""
        pass

    def on_batch_end(self, batch, logs=None):
        """每个batch结束时调用 / Called at the end of each batch"""
        pass

    def on_epoch_end(self, epoch, logs=None):
        """每个epoch结束时调用 / Called at the end of each epoch"""
        pass

    def on_train_end(self, logs=None):
        """训练结束时调用 / Called at the end of training"""
        pass


class CallbackList:
    """
    回调列表 / Container for a list of callbacks

    将多个回调组合在一起，在训练循环中统一调用。
    Combines multiple callbacks and calls them together during training.
    """

    def __init__(self, callbacks=None):
        self.callbacks = callbacks or []

    def on_train_begin(self, logs=None):
        for cb in self.callbacks:
            cb.on_train_begin(logs)

    def on_epoch_begin(self, epoch, logs=None):
        for cb in self.callbacks:
            cb.on_epoch_begin(epoch, logs)

    def on_batch_end(self, batch, logs=None):
        for cb in self.callbacks:
            cb.on_batch_end(batch, logs)

    def on_epoch_end(self, epoch, logs=None):
        for cb in self.callbacks:
            cb.on_epoch_end(epoch, logs)

    def on_train_end(self, logs=None):
        for cb in self.callbacks:
            cb.on_train_end(logs)

print("回调基类定义完成 / Callback base classes defined")

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    训练一个epoch / Train for one epoch

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    device : torch.device
        计算设备 / Compute device

    Returns:
    --------
    float : 平均训练损失 / Average training loss
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches


def evaluate(model, data_loader, criterion, device):
    """
    评估模型 / Evaluate the model

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    data_loader : DataLoader
        数据加载器 / Data loader
    criterion : loss function
        损失函数 / Loss function
    device : torch.device
        计算设备 / Compute device

    Returns:
    --------
    tuple : (平均损失, 平均MAE) / (average loss, average MAE)
    """
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    num_batches = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            mae = nn.L1Loss()(y_pred, y_batch)
            total_loss += loss.item()
            total_mae += mae.item()
            num_batches += 1
    return total_loss / num_batches, total_mae / num_batches


def fit(model, train_loader, valid_loader, criterion, optimizer,
        epochs, device, callbacks=None, lr_scheduler=None):
    """
    带回调的训练循环 / Training loop with callbacks

    这是PyTorch中等价于Keras model.fit()的训练循环，
    集成了回调系统和学习率调度器。

    This is the PyTorch equivalent of Keras model.fit(),
    integrating the callback system and learning rate schedulers.

    Parameters:
    -----------
    model : nn.Module
        PyTorch模型 / PyTorch model
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    criterion : loss function
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    epochs : int
        最大训练轮数 / Maximum number of epochs
    device : torch.device
        计算设备 / Compute device
    callbacks : list of Callback, optional
        回调列表 / List of callbacks
    lr_scheduler : lr_scheduler._LRScheduler, optional
        学习率调度器 / Learning rate scheduler

    Returns:
    --------
    dict : 训练历史 / Training history
    """
    callback_list = CallbackList(callbacks or [])
    history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': [], 'lr': []}

    callback_list.on_train_begin(logs=history)

    stop_training = False
    for epoch in range(epochs):
        if stop_training:
            break

        callback_list.on_epoch_begin(epoch, logs=history)

        # 训练 / Train
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # 评估 / Evaluate
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)

        # 计算训练MAE / Compute training MAE
        _, train_mae = evaluate(model, train_loader, criterion, device)

        # 记录当前学习率 / Record current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        # 保存历史 / Save history
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        history['lr'].append(current_lr)

        # 打印进度 / Print progress
        print(f"Epoch {epoch+1}/{epochs} - "
              f"loss: {train_loss:.4f} - mae: {train_mae:.4f} - "
              f"val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f} - "
              f"lr: {current_lr:.6f}")

        # 学习率调度 / Learning rate scheduling
        if lr_scheduler is not None:
            if isinstance(lr_scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step(val_loss)
            else:
                lr_scheduler.step()

        # 调用epoch结束回调 / Call epoch-end callbacks
        callback_list.on_epoch_end(epoch, logs=history)

        # 检查是否有回调请求停止训练 / Check if any callback requested stop
        for cb in (callbacks or []):
            if hasattr(cb, 'stop_training') and cb.stop_training:
                stop_training = True
                break

    callback_list.on_train_end(logs=history)
    return history

print("训练循环定义完成 / Training loop defined")

## 3. ModelCheckpoint - 模型检查点

自动保存训练过程中的最佳模型。在 PyTorch 中，我们通过保存 `state_dict` 来实现。

Keras 使用 `keras.callbacks.ModelCheckpoint`，PyTorch 需要自己实现，
核心逻辑是：当监控指标改善时，保存模型的 `state_dict`。

In [ ]:
class ModelCheckpoint(Callback):
    """
    模型检查点回调 / Model checkpoint callback

    在监控指标改善时保存模型state_dict。
    等价于Keras的keras.callbacks.ModelCheckpoint。

    Saves model state_dict when the monitored metric improves.
    Equivalent to Keras keras.callbacks.ModelCheckpoint.

    Parameters:
    -----------
    filepath : str
        保存路径 / Save path
    monitor : str
        监控指标名 / Monitored metric name
    save_best_only : bool
        是否只保存最佳 / Whether to save only the best
    mode : str
        'min'表示越小越好，'max'表示越大越好 / 'min' for lower-is-better, 'max' for higher-is-better
    verbose : bool
        是否打印信息 / Whether to print messages
    """

    def __init__(self, filepath, monitor='val_loss', save_best_only=True,
                 mode='min', verbose=True):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.save_best_only = save_best_only
        self.mode = mode
        self.verbose = verbose
        self.best = float('inf') if mode == 'min' else float('-inf')

    def on_train_begin(self, logs=None):
        # 确保目录存在 / Ensure directory exists
        os.makedirs(os.path.dirname(self.filepath) if os.path.dirname(self.filepath) else '.',
                    exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        is_improvement = (self.mode == 'min' and current < self.best) or \
                         (self.mode == 'max' and current > self.best)

        if is_improvement:
            if self.save_best_only:
                self.best = current
                # 保存state_dict / Save state_dict
                torch.save(self.model.state_dict(), self.filepath)
                if self.verbose:
                    print(f"  ModelCheckpoint: {self.monitor} 改善至 {current:.4f}，"
                          f"模型已保存至 {self.filepath}")
        elif not self.save_best_only:
            # 每个epoch都保存 / Save every epoch
            filepath = self.filepath.format(epoch=epoch + 1)
            torch.save(self.model.state_dict(), filepath)


# 使用ModelCheckpoint训练 / Train with ModelCheckpoint
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

checkpoint_cb = ModelCheckpoint(
    filepath='checkpoints_pytorch/best_model.pt',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=True
)
checkpoint_cb.model = model  # 将模型引用传给回调 / Pass model reference to callback

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=30, device=device, callbacks=[checkpoint_cb]
)

In [ ]:
# 加载最佳模型进行评估 / Load best model and evaluate
best_model = create_model()
best_model.load_state_dict(torch.load('checkpoints_pytorch/best_model.pt',
                                       weights_only=True))
best_model.to(device)

test_loss, test_mae = evaluate(best_model, test_loader, criterion, device)
print(f"最佳模型测试集 / Best model on test set - MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 4. EarlyStopping - 早停

当监控指标不再改善时提前停止训练，防止过拟合。

Keras 使用 `keras.callbacks.EarlyStopping`，PyTorch 中需要自己实现：
跟踪 patience 计数器，当连续 `patience` 个 epoch 没有改善时设置 `stop_training = True`。

In [ ]:
class EarlyStopping(Callback):
    """
    早停回调 / Early stopping callback

    当监控指标连续patience个epoch没有改善时停止训练。
    等价于Keras的keras.callbacks.EarlyStopping。

    Stops training when the monitored metric stops improving for
    a specified number of epochs (patience).
    Equivalent to Keras keras.callbacks.EarlyStopping.

    Parameters:
    -----------
    monitor : str
        监控指标名 / Monitored metric name
    patience : int
        等待改善的epoch数 / Number of epochs to wait for improvement
    min_delta : float
        改善阈值 / Minimum change to qualify as improvement
    restore_best_weights : bool
        是否恢复最佳权重 / Whether to restore best weights on stop
    verbose : bool
        是否打印信息 / Whether to print messages
    """

    def __init__(self, monitor='val_loss', patience=10, min_delta=0.0,
                 restore_best_weights=True, verbose=True):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.verbose = verbose
        self.best = float('inf')
        self.best_weights = None
        self.best_epoch = 0
        self.wait = 0
        self.stop_training = False

    def on_train_begin(self, logs=None):
        self.wait = 0
        self.stop_training = False
        self.best = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return

        if current < self.best - self.min_delta:
            # 有改善 / Improvement found
            self.best = current
            self.best_epoch = epoch
            self.wait = 0
            if self.restore_best_weights and self.model is not None:
                self.best_weights = copy.deepcopy(self.model.state_dict())
        else:
            # 没有改善 / No improvement
            self.wait += 1
            if self.wait >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print(f"  EarlyStopping: 连续 {self.patience} 个epoch没有改善，"
                          f"停止训练 / No improvement for {self.patience} epochs, stopping")

    def on_train_end(self, logs=None):
        if self.restore_best_weights and self.best_weights is not None:
            self.model.load_state_dict(self.best_weights)
            if self.verbose:
                print(f"  EarlyStopping: 恢复最佳权重 (epoch {self.best_epoch + 1}) / "
                      f"Restoring best weights (epoch {self.best_epoch + 1})")


# 使用EarlyStopping训练 / Train with EarlyStopping
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=True
)
early_stopping_cb.model = model

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=100,  # 设置较大的epoch数，让早停来决定何时停止 / Large epoch count, let early stopping decide
    device=device, callbacks=[early_stopping_cb]
)

print(f"\n实际训练轮数 / Actual epochs trained: {len(history['loss'])}")

## 5. 组合使用回调函数

结合 ModelCheckpoint 和 EarlyStopping 是最佳实践。我们需要将 `model` 引用传递给每个回调。

In [ ]:
# 组合回调函数 / Combine callbacks
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

callbacks = [
    ModelCheckpoint(
        filepath='checkpoints_pytorch/combined_best.pt',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=True
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=True
    )
]

# 将模型引用传给所有回调 / Pass model reference to all callbacks
for cb in callbacks:
    cb.model = model

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=100, device=device, callbacks=callbacks
)

# 评估 / Evaluate
test_loss, test_mae = evaluate(model, test_loader, criterion, device)
print(f"\n测试集 / Test set - MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 6. ReduceLROnPlateau - 学习率调度

当指标停止改善时自动降低学习率。PyTorch 内置了 `torch.optim.lr_scheduler.ReduceLROnPlateau`，
功能与 Keras 的 `keras.callbacks.ReduceLROnPlateau` 完全等价。

注意：PyTorch 的 `ReduceLROnPlateau` 不是回调，而是 `lr_scheduler`，
需要在训练循环中手动调用 `scheduler.step(val_loss)`。

In [ ]:
# 使用PyTorch内置的ReduceLROnPlateau / Use PyTorch's built-in ReduceLROnPlateau
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# PyTorch内置的学习率调度器 / PyTorch's built-in learning rate scheduler
reduce_lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # 'min'表示监控指标越小越好 / 'min' means lower is better
    factor=0.5,        # 学习率降低因子 / Factor by which LR is reduced
    patience=5,        # 等待改善的epoch数 / Epochs with no improvement
    min_lr=1e-6,       # 最小学习率 / Minimum learning rate
    verbose=True
)

early_stopping_cb = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=True
)
early_stopping_cb.model = model

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=100, device=device,
    callbacks=[early_stopping_cb],
    lr_scheduler=reduce_lr_scheduler  # 传入lr_scheduler / Pass lr_scheduler
)

## 7. LearningRateScheduler - 自定义学习率调度

PyTorch 提供了多种学习率调度器，等价于 Keras 的 `LearningRateScheduler`：

- `LambdaLR`：按自定义函数调整学习率（最灵活，等价于 Keras 的 `LearningRateScheduler`）
- `StepLR`：按固定步长调整
- `CosineAnnealingLR`：余弦退火调度
- `ExponentialLR`：指数衰减

In [ ]:
# 1. LambdaLR - 自定义学习率函数 / Custom learning rate function
# 等价于Keras的LearningRateScheduler / Equivalent to Keras LearningRateScheduler

def exponential_decay_fn(epoch):
    """
    指数衰减学习率调度函数 / Exponential decay learning rate schedule

    前10个epoch保持不变，之后每轮乘以0.9。
    Keep LR unchanged for first 10 epochs, then multiply by 0.9 each epoch.

    Parameters:
    -----------
    epoch : int
        当前epoch / Current epoch

    Returns:
    --------
    float : 学习率乘数 / Learning rate multiplier
    """
    if epoch < 10:
        return 1.0
    else:
        return 0.9 ** (epoch - 10)


model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# LambdaLR接受一个函数，返回当前epoch的学习率乘数
# LambdaLR takes a function that returns LR multiplier for current epoch
lambda_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=exponential_decay_fn)

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=30, device=device, lr_scheduler=lambda_scheduler
)

In [ ]:
# 可视化学习率变化 / Visualize learning rate changes
plt.figure(figsize=(10, 4))
plt.plot(history['lr'])
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (LambdaLR - Exponential Decay)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 2. StepLR - 固定步长衰减 / Step decay
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 每10个epoch学习率乘以0.5 / Multiply LR by 0.5 every 10 epochs
step_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

history_step = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=30, device=device, lr_scheduler=step_scheduler
)

# 可视化 / Visualize
plt.figure(figsize=(10, 4))
plt.plot(history_step['lr'])
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (StepLR)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 3. CosineAnnealingLR - 余弦退火 / Cosine annealing
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 余弦退火：学习率从初始值余弦衰减到eta_min
# Cosine annealing: LR decays from initial value to eta_min following a cosine curve
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30, eta_min=1e-6
)

history_cosine = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=30, device=device, lr_scheduler=cosine_scheduler
)

# 可视化 / Visualize
plt.figure(figsize=(10, 4))
plt.plot(history_cosine['lr'])
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (CosineAnnealingLR)')
plt.grid(True, alpha=0.3)
plt.show()

## 8. 自定义回调函数 - TrainingMonitor

通过继承 `Callback` 基类创建自定义回调，实现训练监控功能。

In [ ]:
class TrainingMonitor(Callback):
    """
    自定义训练监控回调 / Custom training monitor callback

    功能 / Features:
    - 记录每个epoch的训练状态 / Log training status each epoch
    - 在验证损失改善时打印消息 / Print message when val_loss improves
    - 跟踪最佳性能 / Track best performance

    等价于Keras中继承keras.callbacks.Callback的自定义回调。
    Equivalent to a custom callback inheriting keras.callbacks.Callback in Keras.
    """

    def on_train_begin(self, logs=None):
        """训练开始时调用 / Called at the beginning of training"""
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        print("训练开始... / Training started...")

    def on_epoch_end(self, epoch, logs=None):
        """每个epoch结束时调用 / Called at the end of each epoch"""
        logs = logs or {}
        val_loss = logs.get('val_loss')
        if val_loss is not None and val_loss < self.best_val_loss:
            improvement = self.best_val_loss - val_loss
            self.best_val_loss = val_loss
            self.best_epoch = epoch
            print(f"  TrainingMonitor: 验证损失改善 / val_loss improved "
                  f"{improvement:.4f} -> {val_loss:.4f}")

    def on_train_end(self, logs=None):
        """训练结束时调用 / Called at the end of training"""
        print("\n训练完成! / Training complete!")
        print(f"最佳验证损失 / Best val_loss: {self.best_val_loss:.4f} "
              f"(Epoch {self.best_epoch + 1})")


# 使用自定义回调 / Use custom callback
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

custom_cb = TrainingMonitor()
custom_cb.model = model

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=20, device=device, callbacks=[custom_cb]
)

## 9. 可视化训练过程

使用完整的回调组合进行最终训练，并可视化训练曲线。

In [ ]:
# 使用完整回调组合进行最终训练 / Final training with full callback combination
model = create_model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

callbacks = [
    ModelCheckpoint(
        filepath='checkpoints_pytorch/final_best.pt',
        monitor='val_loss',
        save_best_only=True,
        mode='min'
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    TrainingMonitor()
]

for cb in callbacks:
    cb.model = model

reduce_lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)

history = fit(
    model, train_loader, valid_loader, criterion, optimizer,
    epochs=100, device=device, callbacks=callbacks,
    lr_scheduler=reduce_lr_scheduler
)

In [ ]:
# 绘制训练曲线 / Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 损失曲线 / Loss curves
axes[0].plot(history['loss'], label='Training')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE曲线 / MAE curves
axes[1].plot(history['mae'], label='Training')
axes[1].plot(history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 学习率曲线 / Learning rate curve
axes[2].plot(history['lr'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 最终评估 / Final evaluation
test_loss, test_mae = evaluate(model, test_loader, criterion, device)
print(f"\n最终模型测试集 / Final model on test set - MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

## 10. TF vs PyTorch 对照

### 回调系统对照表

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| **回调基类** | `keras.callbacks.Callback` | 自定义 `Callback` 类 |
| **训练循环** | `model.fit(callbacks=[...])` | 自定义 `fit()` 函数 + 手动调用回调 |
| **ModelCheckpoint** | `keras.callbacks.ModelCheckpoint(filepath, monitor, save_best_only)` | 自定义 `ModelCheckpoint` 类，保存 `state_dict` |
| **EarlyStopping** | `keras.callbacks.EarlyStopping(monitor, patience, restore_best_weights)` | 自定义 `EarlyStopping` 类，设置 `stop_training` 标志 |
| **ReduceLROnPlateau** | `keras.callbacks.ReduceLROnPlateau(monitor, factor, patience)` | `torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor, patience)` |
| **LearningRateScheduler** | `keras.callbacks.LearningRateScheduler(schedule_fn)` | `torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)` |
| **自定义回调** | 继承 `keras.callbacks.Callback`，重写 `on_epoch_end` 等 | 继承自定义 `Callback`，重写 `on_epoch_end` 等 |
| **模型保存格式** | `.h5` / SavedModel (完整模型) | `.pt` / `.pth` (state_dict) |
| **恢复最佳权重** | `restore_best_weights=True` | `copy.deepcopy(model.state_dict())` |

### 学习率调度器对照表

| Keras | PyTorch | 说明 |
|-------|---------|------|
| `LearningRateScheduler(fn)` | `LambdaLR(optimizer, lr_lambda=fn)` | 自定义函数调度 |
| 无直接等价 | `StepLR(optimizer, step_size, gamma)` | 固定步长衰减 |
| 无直接等价 | `CosineAnnealingLR(optimizer, T_max)` | 余弦退火 |
| 无直接等价 | `ExponentialLR(optimizer, gamma)` | 指数衰减 |
| `ReduceLROnPlateau` | `ReduceLROnPlateau(optimizer, ...)` | 自适应降低 |
| 无直接等价 | `OneCycleLR(optimizer, max_lr, total_steps)` | 1cycle策略 |

### 关键差异

1. **回调集成方式**：Keras 的 `model.fit()` 内置回调支持；PyTorch 需要自己实现训练循环并手动调用回调
2. **模型保存**：Keras 保存完整模型（结构+权重）；PyTorch 通常只保存 `state_dict`（需要先创建模型再加载）
3. **学习率调度**：Keras 的调度器是回调；PyTorch 的调度器是独立的 `lr_scheduler` 模块，需要在训练循环中调用 `scheduler.step()`
4. **ReduceLROnPlateau 的 step**：PyTorch 版本需要传入监控指标 `scheduler.step(val_loss)`，而 Keras 版本自动获取
5. **灵活性**：PyTorch 的方式更灵活，但需要更多手动代码；Keras 更简洁，但定制性稍弱

## 练习

### 练习1：实现 CosineWarmup 调度器

实现一个带预热的余弦学习率调度器：
- 前 `warmup_epochs` 个 epoch，学习率从 0 线性增长到初始值
- 之后按余弦退火衰减

提示：使用 `LambdaLR`，编写一个返回学习率乘数的函数。

```python
def cosine_warmup_fn(epoch, warmup_epochs=5, total_epochs=50):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # 线性预热
    else:
        # 余弦退火
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
```

### 练习2：实现 GradientClipping 回调

实现一个梯度裁剪回调，在每个 batch 结束后裁剪梯度范数：
- 继承 `Callback` 基类
- 重写 `on_batch_end` 方法（需要修改训练循环以支持 batch 级回调）
- 使用 `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)` 进行裁剪

思考：在 PyTorch 中，梯度裁剪通常直接写在训练循环中，而不是通过回调实现。
这体现了 PyTorch "显式优于隐式" 的设计哲学。

### 练习3：组合回调 + 多种调度器对比

使用相同的模型和数据，分别用以下三种调度策略训练，对比最终测试集性能：
1. `ReduceLROnPlateau`（自适应）
2. `CosineAnnealingLR`（余弦退火）
3. `LambdaLR`（带预热的余弦调度，即练习1的结果）

所有实验都配合 `EarlyStopping(patience=15)` 和 `ModelCheckpoint`，
绘制三种策略的学习率曲线和验证损失曲线进行对比。

## 小结

### 常用回调函数

| 回调 | 功能 | 典型用法 |
|------|------|----------|
| ModelCheckpoint | 保存模型 | 保存最佳模型 |
| EarlyStopping | 早停 | 防止过拟合 |
| ReduceLROnPlateau | 降低学习率 | 突破训练瓶颈 |
| LambdaLR / StepLR / CosineAnnealingLR | 学习率调度 | 实现特定调度策略 |
| TrainingMonitor | 自定义监控 | 跟踪训练状态 |

### 最佳实践

1. **始终使用 ModelCheckpoint**：避免因意外中断丢失进度
2. **结合 EarlyStopping 使用 restore_best_weights**：确保获得最佳模型
3. **设置合理的 patience 值**：太小会过早停止，太大会浪费时间
4. **监控 val_loss 而非 loss**：关注泛化能力而非训练表现
5. **PyTorch 中 lr_scheduler.step() 的调用时机**：在 optimizer.step() 之后、epoch 结束时调用
6. **ReduceLROnPlateau 需要传入监控指标**：`scheduler.step(val_loss)`，其他调度器只需 `scheduler.step()`